In [9]:
from pathlib import Path
from html import escape

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [11]:
# 1. Project paths and settings

try:
    PROJECT_DIR = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_DIR = Path.cwd().parent

DATA_FILE = PROJECT_DIR / "data" / "processed" / "clean_stocks_data.csv"
REPORTS_DIR = PROJECT_DIR / "reports"
FIGURES_DIR = PROJECT_DIR / "figures"

FUNDAMENTALS_FILE = REPORTS_DIR / "fundamental_summary.csv"
TRENDS_FILE = REPORTS_DIR / "annual_fundamental_trends.csv"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = ["AAPL", "MSFT", "NVDA"]
INITIAL_INVESTMENT = 10_000


In [12]:
# 2. Load cleaned historical prices

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. Complete 1 first."
    )

prices = pd.read_csv(DATA_FILE)

required = {"Date", "Ticker", "Close"}
missing = required - set(prices.columns)

if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

prices["Date"] = pd.to_datetime(prices["Date"], errors="coerce")
prices["Ticker"] = prices["Ticker"].astype(str).str.upper().str.strip()
prices["Close"] = pd.to_numeric(prices["Close"], errors="coerce")

prices = prices.dropna(subset=["Date", "Ticker", "Close"])
prices = prices[prices["Close"] > 0]
prices = prices[prices["Ticker"].isin(TICKERS)]
prices = prices.sort_values(["Ticker", "Date"])

if prices.duplicated(["Ticker", "Date"]).any():
    raise ValueError("Duplicate ticker/date rows exist in cleaned data.")

if prices.empty:
    raise ValueError("No valid stock price records were found.")

# Keep only dates shared by all three stocks for fair comparison.
price_table = prices.pivot(
    index="Date", columns="Ticker", values="Close"
)

available_tickers = [
    ticker for ticker in TICKERS if ticker in price_table.columns
]

if len(available_tickers) < 2:
    raise ValueError(
        "At least two stocks are needed for a comparison dashboard."
    )

price_table = price_table[available_tickers].dropna()

if len(price_table) < 2:
    raise ValueError("Not enough common dates to calculate returns.")

# Save the shared date range and price history used in comparisons.
price_table.to_csv(REPORTS_DIR / "dashboard_price_history.csv")


In [13]:
# 3. Calculate comparable performance and risk metrics

daily_returns = price_table.pct_change().dropna(how="all")
summary_rows = []

for ticker in available_tickers:
    series = price_table[ticker].dropna()
    returns = daily_returns[ticker].dropna()

    start_price = series.iloc[0]
    end_price = series.iloc[-1]
    start_date = series.index[0]
    end_date = series.index[-1]

    total_return_pct = (end_price / start_price - 1) * 100

    years = (end_date - start_date).days / 365.25
    cagr_pct = (
        ((end_price / start_price) ** (1 / years) - 1) * 100
        if years > 0 and end_price > 0
        else np.nan
    )

    volatility_pct = (
        returns.std(ddof=1) * np.sqrt(252) * 100
        if len(returns) > 1
        else np.nan
    )

    running_peak = series.cummax()
    drawdown = series / running_peak - 1
    max_drawdown_pct = drawdown.min() * 100

    summary_rows.append({
        "Ticker": ticker,
        "Start_Date": start_date.date(),
        "End_Date": end_date.date(),
        "Starting_Price": start_price,
        "Latest_Price": end_price,
        "Total_Return_Pct": total_return_pct,
        "CAGR_Pct": cagr_pct,
        "Annualized_Volatility_Pct": volatility_pct,
        "Maximum_Drawdown_Pct": max_drawdown_pct,
        "Trading_Days": len(series),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(
    REPORTS_DIR / "dashboard_summary.csv",
    index=False,
)

# Normalize each stock to a $10,000 starting investment.
growth = price_table.div(price_table.iloc[0]).mul(INITIAL_INVESTMENT)
growth.index.name = "Date"
growth.to_csv(REPORTS_DIR / "dashboard_investment_growth.csv")



In [14]:
# 4. Build the investment growth chart

growth_long = (
    growth.reset_index()
    .melt(id_vars="Date", var_name="Ticker", value_name="Value")
)

growth_fig = px.line(
    growth_long,
    x="Date",
    y="Value",
    color="Ticker",
    title=f"Growth of a Hypothetical ${INITIAL_INVESTMENT:,.0f} Investment",
    labels={
        "Date": "Date",
        "Value": "Investment value (USD)",
        "Ticker": "Stock",
    },
)

growth_fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="Stock",
    margin=dict(l=45, r=25, t=75, b=45),
)
growth_fig.write_html(
    FIGURES_DIR / "dashboard_investment_growth.html",
    include_plotlyjs="cdn",
)

In [15]:
# 5. Build the risk-versus-return chart

risk_fig = px.scatter(
    summary,
    x="Annualized_Volatility_Pct",
    y="CAGR_Pct",
    color="Ticker",
    text="Ticker",
    hover_data=[
        "Total_Return_Pct",
        "Maximum_Drawdown_Pct",
        "Trading_Days",
    ],
    title="Historical Annualized Return vs. Volatility",
    labels={
        "Annualized_Volatility_Pct": "Annualized volatility (%)",
        "CAGR_Pct": "CAGR (%)",
        "Ticker": "Stock",
    },
)

risk_fig.update_traces(textposition="top center", marker_size=13)
risk_fig.update_layout(
    template="plotly_white",
    margin=dict(l=45, r=25, t=75, b=45),
)
risk_fig.write_html(
    FIGURES_DIR / "dashboard_risk_return.html",
    include_plotlyjs="cdn",
)


In [ ]:
# 6. Build price and moving-average charts


price_fig = make_subplots(
    rows=len(available_tickers),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=[
        f"{ticker}: Price and Moving Averages"
        for ticker in available_tickers
    ],
)

for row_number, ticker in enumerate(available_tickers, start=1):
    stock = price_table[ticker].dropna()

    ma20 = stock.rolling(20, min_periods=20).mean()
    ma50 = stock.rolling(50, min_periods=50).mean()
    ma200 = stock.rolling(200, min_periods=200).mean()

    price_fig.add_trace(
        go.Scatter(
            x=stock.index,
            y=stock,
            mode="lines",
            name=f"{ticker} adjusted price",
            legendgroup=ticker,
        ),
        row=row_number,
        col=1,
    )

    for moving_average, label in [
        (ma20, "MA20"),
        (ma50, "MA50"),
        (ma200, "MA200"),
    ]:
        price_fig.add_trace(
            go.Scatter(
                x=moving_average.index,
                y=moving_average,
                mode="lines",
                name=f"{ticker} {label}",
                legendgroup=ticker,
            ),
            row=row_number,
            col=1,
        )

price_fig.update_layout(
    template="plotly_white",
    title="Stock Prices and Moving Averages",
    height=330 * len(available_tickers),
    hovermode="x unified",
    legend_title_text="Series (click to show/hide)",
    margin=dict(l=50, r=25, t=85, b=45),
)
price_fig.update_yaxes(title_text="Price (USD)")
price_fig.write_html(
    FIGURES_DIR / "dashboard_moving_averages.html",
    include_plotlyjs="cdn",
)


In [18]:
# 7. Load and chart fundamental data when available

fundamentals = pd.DataFrame()
fundamental_html = (
    "<p>Fundamental summary was not found. "
    "Run Step 7 to generate it.</p>"
)

if FUNDAMENTALS_FILE.exists():
    fundamentals = pd.read_csv(FUNDAMENTALS_FILE)

    if "Ticker" in fundamentals.columns:
        fundamentals = fundamentals[
            fundamentals["Ticker"].astype(str).str.upper().isin(TICKERS)
        ]

        fundamental_html = fundamentals.to_html(
            index=False,
            classes="data-table",
            border=0,
            na_rep="Not available",
            float_format=lambda x: f"{x:,.2f}",
            escape=True,
        )

trend_fig = None

if TRENDS_FILE.exists():
    trends = pd.read_csv(TRENDS_FILE)

    expected = {"Ticker", "Fiscal_Period_End", "Revenue", "Net_Income"}

    if expected.issubset(trends.columns):
        trends["Fiscal_Period_End"] = pd.to_datetime(
            trends["Fiscal_Period_End"], errors="coerce"
        )

        trends["Revenue"] = pd.to_numeric(
            trends["Revenue"], errors="coerce"
        )
        trends["Net_Income"] = pd.to_numeric(
            trends["Net_Income"], errors="coerce"
        )

        trends = trends.dropna(subset=["Fiscal_Period_End"])

        if not trends.empty:
            trend_fig = make_subplots(
                rows=1,
                cols=2,
                subplot_titles=("Annual Revenue", "Annual Net Income"),
            )

            for ticker in TICKERS:
                subset = trends[trends["Ticker"] == ticker].sort_values(
                    "Fiscal_Period_End"
                )

                if subset.empty:
                    continue

                trend_fig.add_trace(
                    go.Scatter(
                        x=subset["Fiscal_Period_End"],
                        y=subset["Revenue"] / 1_000_000_000,
                        mode="lines+markers",
                        name=f"{ticker} revenue",
                    ),
                    row=1,
                    col=1,
                )

                trend_fig.add_trace(
                    go.Scatter(
                        x=subset["Fiscal_Period_End"],
                        y=subset["Net_Income"] / 1_000_000_000,
                        mode="lines+markers",
                        name=f"{ticker} net income",
                    ),
                    row=1,
                    col=2,
                )

            trend_fig.update_yaxes(
                title_text="USD billions", row=1, col=1
            )
            trend_fig.update_yaxes(
                title_text="USD billions", row=1, col=2
            )
            trend_fig.update_layout(
                template="plotly_white",
                title="Company Financial Trends",
                height=480,
                hovermode="x unified",
                margin=dict(l=45, r=25, t=85, b=45),
            )


In [19]:
# 8. Create KPI cards and a stock summary table

best_return = summary.loc[summary["Total_Return_Pct"].idxmax()]
highest_volatility = summary.loc[
    summary["Annualized_Volatility_Pct"].idxmax()
]

def money(value):
    return f"${value:,.2f}" if pd.notna(value) else "N/A"

def percentage(value):
    return f"{value:.2f}%" if pd.notna(value) else "N/A"

cards = [
    ("Stocks compared", str(len(summary))),
    ("Best historical total return", f"{best_return['Ticker']} · {percentage(best_return['Total_Return_Pct'])}"),
    ("Highest annualized volatility", f"{highest_volatility['Ticker']} · {percentage(highest_volatility['Annualized_Volatility_Pct'])}"),
    ("Latest common date", str(price_table.index[-1].date())),
]

card_html = "\n".join(
    f'<div class="card"><div class="card-label">{escape(label)}</div>'
    f'<div class="card-value">{escape(value)}</div></div>'
    for label, value in cards
)

summary_display = summary.copy()

for column in [
    "Starting_Price",
    "Latest_Price",
    "Total_Return_Pct",
    "CAGR_Pct",
    "Annualized_Volatility_Pct",
    "Maximum_Drawdown_Pct",
]:
    if column in summary_display.columns:
        summary_display[column] = pd.to_numeric(
            summary_display[column], errors="coerce"
        ).round(2)

summary_html = summary_display.to_html(
    index=False,
    classes="data-table",
    border=0,
    na_rep="Not available",
    escape=True,
)

In [20]:
# 9. Assemble the interactive HTML dashboard

figures = [
    ("Investment Growth", growth_fig),
    ("Risk and Return", risk_fig),
    ("Moving Averages", price_fig),
]

if trend_fig is not None:
    figures.append(("Fundamentals", trend_fig))

chart_sections = []

for index, (title, figure) in enumerate(figures):
    chart_html = figure.to_html(
        full_html=False,
        include_plotlyjs=True if index == 0 else False,
        config={"responsive": True, "displaylogo": False},
    )

    chart_sections.append(
        f'<section class="panel"><h2>{escape(title)}</h2>{chart_html}</section>'
    )

if trend_fig is None:
    chart_sections.append(
        '<section class="panel"><h2>Fundamental Trends</h2>'
        '<p>Annual revenue and net income charts are unavailable. '
        'Run Step 7 and confirm reports/annual_fundamental_trends.csv '
        'contains Ticker, Fiscal_Period_End, Revenue, and Net_Income.</p>'
        '</section>'
    )

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Stock Market Analysis Dashboard</title>
<style>
    body {{
        margin: 0;
        background: #f3f6fa;
        color: #172033;
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    }}
    header {{
        background: #14243a;
        color: white;
        padding: 32px 5%;
    }}
    header h1 {{ margin: 0 0 8px; font-size: 28px; }}
    header p {{ margin: 0; color: #d5deeb; }}
    main {{ max-width: 1400px; margin: auto; padding: 24px; }}
    .cards {{
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(210px, 1fr));
        gap: 16px;
        margin-bottom: 24px;
    }}
    .card, .panel {{
        background: white;
        border-radius: 12px;
        padding: 20px;
        box-shadow: 0 2px 10px rgba(20, 36, 58, 0.06);
        margin-bottom: 20px;
    }}
    .card-label {{ color: #5b6575; font-size: 13px; margin-bottom: 10px; }}
    .card-value {{ font-size: 21px; font-weight: 700; overflow-wrap: anywhere; }}
    h2 {{ font-size: 19px; margin-top: 0; }}
    .data-table {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
    .data-table th, .data-table td {{
        text-align: left; padding: 10px; border-bottom: 1px solid #e7ebf0;
    }}
    .data-table th {{ background: #f7f9fc; }}
    .table-wrap {{ overflow-x: auto; }}
    .note {{ color: #5b6575; font-size: 13px; line-height: 1.6; }}
    footer {{ padding: 24px 5%; color: #5b6575; font-size: 12px; }}
</style>
</head>
<body>
<header>
    <h1>Stock Market Analysis Dashboard</h1>
    <p>Apple (AAPL) · Microsoft (MSFT) · NVIDIA (NVDA)</p>
</header>
<main>
    <div class="cards">{card_html}</div>

    <section class="panel">
        <h2>Performance and Risk Summary</h2>
        <div class="table-wrap">{summary_html}</div>
    </section>

    {''.join(chart_sections)}

    <section class="panel">
        <h2>Fundamental Summary</h2>
        <div class="table-wrap">{fundamental_html}</div>
    </section>

    <section class="panel">
        <h2>How to interpret this dashboard</h2>
        <p class="note">
        Total return and CAGR describe historical price growth over the shared
        date range. Volatility measures the variability of daily returns.
        Maximum drawdown measures the largest peak-to-trough decline.
        Moving averages help describe trends, but do not predict future prices.
        Fundamental metrics are reported on company fiscal periods and may not
        have the same dates as market metrics. Historical performance does not
        guarantee future results.
        </p>
    </section>
</main>
<footer>
    Data source: local project datasets originally collected for this analysis.
    Generated with Python, pandas, NumPy, and Plotly.
</footer>
</body>
</html>
"""

dashboard_file = REPORTS_DIR / "stock_market_dashboard.html"
dashboard_file.write_text(html, encoding="utf-8")

print("\nDashboard created successfully.")
print(f"Open this file in your browser:\n{dashboard_file}")
print(f"Summary CSV:\n{REPORTS_DIR / 'dashboard_summary.csv'}")
print(f"Investment growth CSV:\n{REPORTS_DIR / 'dashboard_investment_growth.csv'}")



Dashboard created successfully.
Open this file in your browser:
/Users/golden_horde/Desktop/FA_projects/stock-market-analysis/reports/stock_market_dashboard.html
Summary CSV:
/Users/golden_horde/Desktop/FA_projects/stock-market-analysis/reports/dashboard_summary.csv
Investment growth CSV:
/Users/golden_horde/Desktop/FA_projects/stock-market-analysis/reports/dashboard_investment_growth.csv
